### Embedding using Gemini Embedding model


In [1]:
%pip install -qU langchain-unstructured

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -qU langchain-experimental

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -qU unstructured

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -qU rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_unstructured import UnstructuredLoader
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
import os
from dotenv import load_dotenv
from langchain.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever

# Load environment variables from a .env file if present
load_dotenv()

# Set required environment variables
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 1. Initialize Gemini embedding model
embedding_model = "models/text-embedding-004" 
embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model, api_key=GOOGLE_API_KEY)


# 2. Load sample text from a web page
page_url = "https://oauife.edu.ng/admission-undergraduate-studies/"
loader = UnstructuredLoader(web_url=page_url)

docs = []
async for doc in loader.alazy_load():
    docs.append(doc)

# 3. Chunking using SemanticChunker (Gemini embedding model)
chunker = SemanticChunker(embeddings=embeddings, breakpoint_threshold_type='gradient', breakpoint_threshold_amount=0.8)
chunks = chunker.split_documents(docs)
for doc in chunks[10:20]:
    print(doc.page_content)

# 4. Connect to Qdrant (local or cloud)
client = QdrantClient(':memory:')  # Or use your cloud URL and API key

collection_name = "test_gemini"
vector_size = 768

# 5. Create collection if not exists
if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )

# 6. Create vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# 7. Embed and add documents
vector_store.add_documents(chunks)
print("✅ Documents embedded and stored in Qdrant.")

# 8. Test retrieval

query = "Which UME subjects do I need to study computer science with economics in OAU?"

bm25_retriever = BM25Retriever.from_documents(chunks, k=5)

similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

# Create an ensemble retriever that combines similarity and BM25 retrievers
ensemble_retriever = EnsembleRetriever(retrievers=[similarity_retriever, bm25_retriever], weights=[0.5, 0.5])

# Retrieve top 5 results using the ensemble retriever
ensemble_results = ensemble_retriever.invoke(query)
print("\nTop results for query (Ensemble):")
for i, doc in enumerate(ensemble_results):
    print(f"{i+1}. {doc.page_content}")


# 9. Clean up (optional)
# client.delete_collection(collection_name=collection_name)




S/N Course/Program UME Requirements O/Level Requirements Direct Entry Requirements 1. Pharmacy English, Physics Chemistry and Biology English Language, Mathematics, Physics  Chemistry and Biology B.Sc. with a minimum of 2 2 in Biochemistry, Microbiology Industrial Chemistry, Food Science & Technology
OBAFEMI AWOLOWO UNIVERSITY, ILE-IFE, NIGERIA
Requirements for Admission into Part I and Direct Entry in the Faculties
COLEGE OF HEALTH SCIENCES
S/N Course/Program UME Requirements O/Level Requirements Direct Entry Requirements a.
Medicine English, Physics Chemistry and Biology English Language, Mathematics, Physics  Chemistry and Biology (i)  B.Sc.
with a minimum of 2 1 in Biology, Chemistry, Anatomy, Biochemistry, Physiology and other Biological Sciences in which   their student take exactly the same courses  in their first year as  first year student of Medicine and Dentistry (ii)  3 A’level passes in Biology, Chemistry, Physics b.
Dentistry English, Physics Chemistry and Biology English

In [13]:
import re

def condense_query(query):
    # Remove common stopwords and question words
    stopwords = {"which", "do", "i", "need", "to", "study", "in", "the", "a", "is", "for", "of", "and"}
    words = re.findall(r'\w+', query.lower())
    keywords = [w for w in words if w not in stopwords]
    return " ".join(keywords)

# Use condensed query for retrieval
condensed = condense_query(query)
ensemble_results = ensemble_retriever.invoke(condensed)

print("\nTop results for query (Ensemble):")
for i, doc in enumerate(ensemble_results):
    print(f"{i+1}. {doc.page_content}")


Top results for query (Ensemble):
1. Computer Science with Economics English, Chemistry, Physics and Maths English, Mathematics, Physics, Chemistry and Biology/Agric.
2. Computer Science with Mathematics English, Chemistry, Physics and Maths English, Mathematics, Physics, Chemistry and Biology/Agric.
3. Education (Religious Studies) 2 Arts subjects including Religious and 1 other subject English Language, IRS/CRS, plus any other three subjects from Literature in English, Yoruba/Hausa/Igbo, History/ Government, French, Economics, Geography, Biology, Agric.
4. Economics English, Mathematics, Government and Economics English, Mathematics, Economics 1 from Government/Geography and one from Biology, Agriculture,  Civics, History, IRK/CRK, Yoruba, Literature, Chemistry, Physics, Further Mathematics and ICT (i)                 3 A’ Level Passes in Economics, Government and Mathematics (ii)               Upper Credit Passes in ND/HND in Economics b.
5. Computer Engineering English, Chemistry,

In [14]:
from langchain_community.document_transformers import (
    LongContextReorder,
)

reordering = LongContextReorder()
reordered_docs = reordering.transform_documents(ensemble_results)

print("\nReordered documents:")
for i, doc in enumerate(reordered_docs):
    print(f"{i+1}. {doc.page_content}") # Print first 100 chars for brevity


Reordered documents:
1. Computer Science with Mathematics English, Chemistry, Physics and Maths English, Mathematics, Physics, Chemistry and Biology/Agric.
2. Economics English, Mathematics, Government and Economics English, Mathematics, Economics 1 from Government/Geography and one from Biology, Agriculture,  Civics, History, IRK/CRK, Yoruba, Literature, Chemistry, Physics, Further Mathematics and ICT (i)                 3 A’ Level Passes in Economics, Government and Mathematics (ii)               Upper Credit Passes in ND/HND in Economics b.
3. Science, Principles of Accounts, Government, Civic Education, Music Diploma in Yoruba/NCE with Language option j.
4. German English Plus 4 other subjects from Literature/Yoruba/ Igbo/CRS/IRS/History/French/Government/ Fine Arts/ Mathematics/Music/Economic/Physics/Chemistry/Biology Home Economics/Geography/Food & Nutrition/Civic Education, Agric.
5. Education (Economics) Economics, Mathematics, and other subject from Geography/ Physics, Histor

### Metadata Extraction via Gemini (LLMChain)

In [ ]:
import json
from datetime import datetime, timezone
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Use Gemini LLM (already initialized as `llm`)
metadata_prompt = PromptTemplate(input_variables=["chunk"],
    template="""
You're extracting metadata from Nigerian education documents for a student info bot.
Analyze the chunk below and extract structured metadata in JSON.

Chunk:
---
{chunk}
---

Return only JSON with fields:
- source_type (e.g., JAMB/UTME brochure, handbook, etc.)
- institution (e.g: UTME, NUC, OAU, UNILAG, etc.)
- faculty
- department
- academic_year
- section_title
- document_topic (e.g., admission, fees, requirements, etc.)
- education_level
- admission_type (if applicable)
- uploaded_at (ISO timestamp, use current time if not present)
- page_number

JSON:
""")

# Compose the chain: prompt -> LLM -> output parser
metadata_chain = metadata_prompt | llm | StrOutputParser()

# Function to extract metadata from a document chunk using the LLM
def extract_metadata_llm(chunk: str, page_number=None, uploaded_at=None):
    try:
        metadata_json = metadata_chain.invoke({"chunk": chunk})
        if not metadata_json:
            print(f"No metadata returned for page {page_number}")
            return {
                "page_number": page_number or -1,
                "uploaded_at": uploaded_at or datetime.now(timezone.utc).isoformat()
            }
        
        # Strip markdown backticks if present
        metadata_json = metadata_json.strip().strip("`").strip()
        if metadata_json.startswith("json"):
            metadata_json = metadata_json[len("json"):].strip()

    except Exception as e:
        print(f"Failed to run metadata_chain on page {page_number}: {e}")
        return {
            "page_number": page_number or -1,
            "uploaded_at": uploaded_at or datetime.now(timezone.utc).isoformat()
        }

    try:
        metadata = json.loads(metadata_json)
    except json.JSONDecodeError as e:
        print(f"Metadata JSON parse error (page {page_number}): {e}")
        metadata = {}

    # Fallback metadata
    metadata["page_number"] = page_number or -1
    metadata["uploaded_at"] = uploaded_at or datetime.now(timezone.utc).isoformat()

    print(f"📝 Parsed metadata for page {page_number}: {metadata}\n")
    return metadata


### Smart Loader + Metadata Extraction in One
This extracts the section then runs LLM calls per section to reduce LLM calls per documents

In [ ]:
from langchain_core.documents import Document
from tqdm import tqdm
from datetime import datetime, timezone
import re

def split_text_by_section(text: str):
    """
    Splits the full document text into natural sections based on headings.
    This version assumes headings are either numbered (1., 2.1, etc.) or uppercase section titles.
    """
    section_pattern = re.compile(r"(?=\n(?:\d{1,2}[\.\)]\s+|[A-Z][A-Z\s]{3,}))")
    sections = section_pattern.split(text)
    return [s.strip() for s in sections if s.strip()]

def process_docs_with_metadata(docs, uploaded_at=None, force_per_chunk=False):
    final_docs = []
    uploaded_at = uploaded_at or datetime.now(timezone.utc).isoformat()

    if force_per_chunk:
        print("⚙️ Using per-chunk metadata extraction (forced).")
    else:
        # Try section-based
        full_text = "\n\n".join(doc.page_content for doc in docs)
        sections = split_text_by_section(full_text)
        print(f"📖 Detected {len(sections)} sections in full text.")

        if len(sections) > 0:
            section_metadata = []
            print("⏳ Extracting metadata per section...")
            for i, section in enumerate(tqdm(sections, desc="Section Metadata")):
                meta = extract_metadata_llm(section, page_number=i, uploaded_at=uploaded_at)
                section_metadata.append(meta)

            # If we successfully extracted anything meaningful, continue
            if any(bool(m) and len(m) > 2 for m in section_metadata):  # crude check
                chunks_per_section = len(docs) // max(len(section_metadata), 1)

                for i, doc in enumerate(docs):
                    sec_idx = min(i // chunks_per_section, len(section_metadata) - 1)
                    chunk_meta = section_metadata[sec_idx].copy()
                    chunk_meta["page_number"] = doc.metadata.get("page_number", i)
                    final_docs.append(Document(page_content=doc.page_content, metadata=chunk_meta))
                return final_docs
            else:
                print("⚠️ Section metadata extraction failed or returned empty. Falling back to per-chunk.")

    # 🛑 Fallback to per-chunk extraction
    print("⏳ Extracting metadata per chunk...")
    for i, doc in enumerate(tqdm(docs, desc="Chunk Metadata")):
        try:
            meta = extract_metadata_llm(doc.page_content, page_number=i, uploaded_at=uploaded_at)
            final_docs.append(Document(page_content=doc.page_content, metadata=meta))
        except Exception as e:
            print(f"❌ Failed to process chunk {i}: {e}")
    return final_docs
